### importamos librerias

In [1]:
import cv2

#### si queremos detectar personas, importamos yolo  desde ultralytics

In [2]:
from ultralytics import YOLO

#### si queremos detectar apriltags importamos pupil_apriltags

In [3]:
from pupil_apriltags import Detector

#### si queremos detectar gestos, importamos mediapipe

In [4]:
import mediapipe as mp

### Lectura de una imagen 

Podemos leer una imagen, como podemos leer un video.

Para leer una imagen, podemos usar la función `cv2.imread()`.


In [5]:
frame = cv2.imread('imagetest.png')

Para mostrar la imagen, usamos `cv2.imshow('imagen',imagen)`

In [6]:
cv2.imshow("Imagen", frame)
cv2.waitKey(1000)
cv2.destroyAllWindows()
cv2.waitKey(1) # esto no es necesario, pero en jupyter funciona, no tengo claro por qué

-1

# APRILTAGS

### si queremos detectar un apriltag, creamos el detector

In [7]:
at_detector = Detector(
   families="tag36h11", # la familia
   nthreads=1,
   quad_decimate=1.0,  # downscale factor
   quad_sigma=0.0, # blur sigma 
   refine_edges=1, #refine (? )
   decode_sharpening=0.25, # 
   debug=0
)

#### pupil apriltags detecta en escala de grises

In [8]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

#### Usamos el detector

In [9]:
apriltags = at_detector.detect(gray)

#### procesamos los apriltags

#### 

In [ ]:
# obtenemos los resultados de todos los apriltags
for apriltag in apriltags:
    corners = apriltag.corners.astype(int)

    # DIBUJAR BOUNDING BOX
    for i in range(4):
        pt1 = tuple(corners[i])
        pt2 = tuple(corners[(i+1) % 4])
        cv2.line(frame, pt1, pt2, (0,255,0), 2)

        # OBTENEMOS EL CENTRO
        center = apriltag.center
        cx, cy = int(center[0]), int(center[1])
        cv2.circle(frame, (cx, cy), 5, (0,0,255), -1)
        

In [17]:
cv2.imshow("Imagen", frame)
cv2.waitKey(5000)
cv2.destroyAllWindows()
cv2.waitKey(1) # esto no es necesario, pero en jupyter funciona, no tengo claro por qué

-1

# DETECCION PERSONAS

Importamos los pesos entrenados por yolo

In [19]:
model = YOLO("yolo26m.pt")

generamos predicción

In [20]:
personas = model.predict(frame, device='cpu', verbose=False)


#### Procesamos cada detección 

- box = la caja que nos encierra a la persona 
- cls = la clase de la persona
- conf = la confianza de que la persona sea la persona que estamos buscando 
- label = la etiqueta de la clase de la persona

#### por otra parte, x1, y1 son las coordenadas de la imagen en la esquina superior izquierda, 

#### Mientras que x2, y2 son las coordenadas de la esquina inferior derecha.


In [ ]:
for persona in personas:

    for box, cls, conf in zip(persona.boxes.xyxy, persona.boxes.cls, persona.boxes.conf):

        x1, y1, x2, y2 = map(int, box)
        label = persona.names[int(cls)]
        text = f"{label} {conf:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


#### Usamos `cv2.rectangle` para dibujar los cuadros de la imagen

#### Usamos `cv2.putText` para dibujar el texto

In [24]:
cv2.imshow("Imagen", frame)
cv2.waitKey(15000)
cv2.destroyAllWindows()
cv2.waitKey(1) # esto no es necesario, pero en jupyter funciona, no tengo claro por qué

-1

# VIDEO 

In [31]:
import os 

images = sorted(os.listdir("images"))
print(images)

['000000.png', '000001.png', '000002.png', '000003.png', '000004.png', '000005.png', '000006.png', '000007.png', '000008.png', '000009.png', '000010.png', '000011.png', '000012.png', '000013.png', '000014.png', '000015.png', '000016.png', '000017.png', '000018.png', '000019.png', '000020.png', '000021.png', '000022.png', '000023.png', '000024.png', '000025.png', '000026.png', '000027.png', '000028.png', '000029.png', '000030.png', '000031.png', '000032.png', '000033.png', '000034.png', '000035.png', '000036.png', '000037.png', '000038.png', '000039.png', '000040.png', '000041.png', '000042.png', '000043.png', '000044.png', '000045.png', '000046.png', '000047.png', '000048.png', '000049.png', '000050.png', '000051.png', '000052.png', '000053.png', '000054.png', '000055.png', '000056.png', '000057.png', '000058.png', '000059.png', '000060.png', '000061.png', '000062.png', '000063.png', '000064.png', '000065.png', '000066.png', '000067.png', '000068.png', '000069.png', '000070.png', '0000

In [ ]:
fps = 15
index = 0
while True:
    frame = cv2.imread(os.path.join('images', images[index]))

    cv2.imshow("Video", frame)
    key = cv2.waitKey(100) & 0xFF


    


    if key == ord('q'):
        break
    
    index = (index + 1) % len(images)

## PODEMOS DETECTAR APRILTAGS

In [34]:
fps = 15
index = 0
while True:
    frame = cv2.imread(os.path.join('images', images[index]))



    #### ========= DETECTOR DE APRILTAGS =========

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    apriltags = at_detector.detect(gray)
                
    # obtenemos los resultados de todos los apriltags
    for apriltag in apriltags:
        corners = apriltag.corners.astype(int)

        # DIBUJAR BOUNDING BOX
        for i in range(4):
            pt1 = tuple(corners[i])
            pt2 = tuple(corners[(i+1) % 4])
            cv2.line(frame, pt1, pt2, (0,255,0), 2)

            # OBTENEMOS EL CENTRO
            center = apriltag.center
            cx, cy = int(center[0]), int(center[1])
            cv2.circle(frame, (cx, cy), 5, (0,0,255), -1)
        

    cv2.imshow("Video", frame)
    key = cv2.waitKey(100) & 0xFF
    if key == ord('q'):
        break
    
    index = (index + 1) % len(images)

#### y tambien podemos detectar personas 

In [ ]:
fps = 15
index = 0
while True:
    frame = cv2.imread(os.path.join('images', images[index]))


    #### ========= DETECTOR DE PERSONAS =========
    personas = model.predict(frame, device='cpu', verbose=False)
    for persona in personas:
        for box, cls, conf in zip(persona.boxes.xyxy, persona.boxes.cls, persona.boxes.conf):
            x1, y1, x2, y2 = map(int, box)
            label = persona.names[int(cls)]
            text = f"{label} {conf:.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


    cv2.imshow("Video", frame)
    key = cv2.waitKey(100) & 0xFF
    if key == ord('q'):
        break
    
    index = (index + 1) % len(images)

#### y podemos mezclar los dos 

In [35]:
fps = 15
index = 0
while True:
    frame = cv2.imread(os.path.join('images', images[index]))



    #### ========= DETECTOR DE APRILTAGS =========

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    apriltags = at_detector.detect(gray)
                
    # obtenemos los resultados de todos los apriltags
    for apriltag in apriltags:
        corners = apriltag.corners.astype(int)

        # DIBUJAR BOUNDING BOX
        for i in range(4):
            pt1 = tuple(corners[i])
            pt2 = tuple(corners[(i+1) % 4])
            cv2.line(frame, pt1, pt2, (0,255,0), 2)

            # OBTENEMOS EL CENTRO
            center = apriltag.center
            cx, cy = int(center[0]), int(center[1])
            cv2.circle(frame, (cx, cy), 5, (0,0,255), -1)
        


    #### ========= DETECTOR DE PERSONAS =========
    personas = model.predict(frame, device='cpu', verbose=False)
    for persona in personas:
        for box, cls, conf in zip(persona.boxes.xyxy, persona.boxes.cls, persona.boxes.conf):
            x1, y1, x2, y2 = map(int, box)
            label = persona.names[int(cls)]
            text = f"{label} {conf:.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


    cv2.imshow("Video", frame)
    key = cv2.waitKey(100) & 0xFF
    if key == ord('q'):
        break
    
    index = (index + 1) % len(images)